<a href="https://colab.research.google.com/github/GustavoNachbar/churn-dataset-clusters-classify-tests/blob/main/k_medoids.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install scikit-learn-extra

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.0/819.0 kB 34.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for scikit-learn-extra: filename=scikit_learn_extra-0.3.0-cp313-cp313-linux_x86_64.whl size=2154566 sha256=327ea03991a6e7d5945be4d0a1e44e8c01c13a95e210e1dbaaf70b5d14fbe8d2
  Stored in directory: /root/.cache/pip/wheels/17/4b/b8/6b6711681d0981b110c9cc91ad6d1ebd88adf1547e1da301fc
Successfully built scikit-learn-extra


In [3]:
from sklearn_extra.cluster import KMedoids
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score
from sklearn.utils import resample
import numpy as np
import pandas as pd

df = pd.read_csv("/content/Churn_Modelling.csv")

In [4]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [5]:
from sklearn.model_selection import train_test_split

# 70% treino, 30% temporário
df_treino, df_temp = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["Exited"]
)

# 15% teste, 15% validação
df_teste, df_validacao = train_test_split(
    df_temp,
    test_size=0.50,
    random_state=42,
    stratify=df_temp["Exited"]
)

# Verificando os tamanhos
print(f"Treino: {len(df_treino)} ({len(df_treino)/len(df):.1%})")
print(f"Teste: {len(df_teste)} ({len(df_teste)/len(df):.1%})")
print(f"Validação: {len(df_validacao)} ({len(df_validacao)/len(df):.1%})")

# Verificando a distribuição de Exited
print("\nDistribuição no DF original:")
print(df["Exited"].value_counts(normalize=True))

print("\nDistribuição no treino:")
print(df_treino["Exited"].value_counts(normalize=True))

print("\nDistribuição no teste:")
print(df_teste["Exited"].value_counts(normalize=True))

print("\nDistribuição na validação:")
print(df_validacao["Exited"].value_counts(normalize=True))

Treino: 7000 (70.0%)
Teste: 1500 (15.0%)
Validação: 1500 (15.0%)

Distribuição no DF original:
Exited
0    0.7963
1    0.2037
Name: proportion, dtype: float64

Distribuição no treino:
Exited
0    0.796286
1    0.203714
Name: proportion, dtype: float64

Distribuição no teste:
Exited
0    0.796
1    0.204
Name: proportion, dtype: float64

Distribuição na validação:
Exited
0    0.796667
1    0.203333
Name: proportion, dtype: float64


In [6]:
df_treino = df_treino.rename(columns={
    'RowNumber': 'NumeroLinha',
    'CustomerId': 'IdCliente',
    'Surname': 'Sobrenome',
    'CreditScore': 'PontuacaoCredito',
    'Geography': 'Geografia',
    'Gender': 'Genero',
    'Age': 'Idade',
    'Tenure': 'TempoRelacionamento',
    'Balance': 'Saldo',
    'NumOfProducts': 'NumeroProdutos',
    'HasCrCard': 'PossuiCartaoCredito',
    'IsActiveMember': 'MembroAtivo',
    'EstimatedSalary': 'SalarioEstimado',
    'Exited': 'Saiu'
})

In [10]:
from sklearn.preprocessing import StandardScaler

variaveis_numericas = [
    'PontuacaoCredito', 'Idade', 'TempoRelacionamento',
    'Saldo', 'NumeroProdutos', 'SalarioEstimado'
]

df_treino_cancelamento = df_treino[df_treino["Saiu"] == 1][variaveis_numericas]

# Scaler específico para a base de cancelados (não reaproveitar o scaler do df_treino completo)
scaler_cancelamento = StandardScaler()
X_scaled = scaler_cancelamento.fit_transform(df_treino_cancelamento)

In [11]:
from sklearn_extra.cluster import KMedoids
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, adjusted_rand_score
from sklearn.utils import resample
import numpy as np
import pandas as pd

def calcular_estabilidade_generico(X, modelo_classe, modelo_kwargs, labels_referencia,
                                     n_bootstrap=30, sample_frac=0.8, random_state=42):
    rng = np.random.default_rng(random_state)
    X_arr = np.asarray(X)
    n_samples = X_arr.shape[0]
    ari_scores = []

    for i in range(n_bootstrap):
        seed = int(rng.integers(0, 1_000_000))
        idx_sample = resample(
            np.arange(n_samples), replace=True,
            n_samples=int(n_samples * sample_frac), random_state=seed
        )
        idx_sample = np.unique(idx_sample)
        X_sample = X_arr[idx_sample]

        modelo_boot = modelo_classe(**modelo_kwargs, random_state=seed)
        labels_boot = modelo_boot.fit_predict(X_sample)

        labels_ref_sample = np.asarray(labels_referencia)[idx_sample]
        ari_scores.append(adjusted_rand_score(labels_ref_sample, labels_boot))

    return np.mean(ari_scores), np.std(ari_scores)


resultados_kmedoids = []

# X_scaled aqui é o gerado a partir do df_treino_cancelamento (célula acima)
for k in range(2, 8):
    modelo = KMedoids(n_clusters=k, random_state=42, method='alternate', init='k-medoids++')
    labels = modelo.fit_predict(X_scaled)

    estab_media, estab_std = calcular_estabilidade_generico(
        X_scaled, KMedoids,
        {'n_clusters': k, 'method': 'alternate', 'init': 'k-medoids++'},
        labels
    )

    resultados_kmedoids.append({
        'k': k,
        'silhouette': silhouette_score(X_scaled, labels),
        'davies_bouldin': davies_bouldin_score(X_scaled, labels),
        'calinski_harabasz': calinski_harabasz_score(X_scaled, labels),
        'estabilidade_media_ari': estab_media,
        'estabilidade_std_ari': estab_std,
    })

df_resultados_kmedoids = pd.DataFrame(resultados_kmedoids).set_index('k')
df_resultados_kmedoids

,silhouette,davies_bouldin,calinski_harabasz,estabilidade_media_ari,estabilidade_std_ari
k,,,,,
2,0.117436,2.677723,184.023363,0.065492,0.080562
3,0.097106,2.361304,159.316680,0.123451,0.066772
4,0.102904,2.073692,160.238872,0.162228,0.050086
5,0.115907,1.888967,164.824707,0.233701,0.064986
6,0.109108,1.775885,158.504228,0.261496,0.051858
7,0.120043,1.770965,157.931179,0.260272,0.055371
